# KYC - field detector training (Kaggle GPU -> ONNX)

Pulls the labelled cards, rectifies them, trains the field detector and
exports it to ONNX for CPU serving. One run, start to finish.

**Before you run:**

1. Settings > Accelerator > **GPU T4 x2**
2. Settings > Internet > **On**
3. Add-ons > Secrets > add **`ROBOFLOW_API_KEY`** (only needed when
   `DATA_SOURCE = "roboflow"`)

The exported `.onnx` lands in `/kaggle/working/models/`. Fetch it with
`python -m scripts.kaggle_run pull`, or download it from the Output tab.


## 1. Configuration - the only cell you edit

In [ ]:
# Where the labelled cards come from:
#   "roboflow"       - pull them here, inside the kernel. Use this when your
#                      own network cannot reach Roboflow's CDN.
#   "kaggle-dataset" - use a dataset you already attached under Add Data.
DATA_SOURCE = "roboflow"

# --- DATA_SOURCE = "roboflow" ---------------------------------------
RF_WORKSPACE = "westco"
RF_PROJECT   = "idcardsegmentation-gz97d"
RF_VERSION   = 8
RF_FORMAT    = "yolov8"

# --- DATA_SOURCE = "kaggle-dataset" ---------------------------------
ATTACHED_DATA_YAML = "/kaggle/input/CHANGE-ME/data.yaml"

# The detector runs on cards the pipeline has already flattened, so the
# training set has to be flattened the same way. Leave this on unless the
# dataset is already rectified.
RECTIFY = True

PROFILE = "ir_national_card_front"
BASE    = "yolov8n.pt"       # n is enough once the card is rectified
EPOCHS  = 150
IMGSZ   = 640
BATCH   = 16

# Source of the kyc package. scripts/kaggle_run.py uploads it as a private
# dataset and attaches it here.
SRC_DATASET = "/kaggle/input/kyc-src"

# Falling back to a git clone is opt-in. It runs whatever is on the repo's
# default branch, which is not necessarily the code you just pushed.
ALLOW_GIT_CLONE = False
REPO_URL        = "https://github.com/zhaleh197/kyc.git"


## 2. Dependencies

In [ ]:
# pydantic-settings because the training script imports the document profile
# from the kyc package - that is the point, so the class order used here cannot
# drift from the class order the serving pipeline expects.
!pip install -q ultralytics onnx onnxsim pydantic-settings roboflow

import torch
import ultralytics

print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


## 3. Get the source

The training script and the document profiles come from the repo, so nothing
about the card layout is duplicated into this notebook.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working/src")
INPUT = Path("/kaggle/input")

print("attached under /kaggle/input:")
for entry in sorted(INPUT.iterdir()) if INPUT.exists() else []:
    print("   ", entry.name)

if Path(SRC_DATASET).exists():
    print("\nusing attached source dataset:", SRC_DATASET)
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(SRC_DATASET, WORK)
elif ALLOW_GIT_CLONE:
    # Opt-in only. Falling back to the published repo silently is worse than
    # failing: it runs whatever code happens to be on the default branch, which
    # is how a run once failed on a module that only existed locally.
    print("\nsource dataset missing; cloning", REPO_URL)
    if not WORK.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
else:
    raise SystemExit(
        f"Expected the source dataset at {SRC_DATASET} but it is not attached.\n"
        "Push it with:  python -m scripts.kaggle_run push\n"
        "Or set ALLOW_GIT_CLONE = True to run whatever is on the repo's default "
        "branch instead - only do that if you know it is current."
    )

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())


PROGRESS = Path("/kaggle/working/progress")
PROGRESS.mkdir(parents=True, exist_ok=True)


def mark(step, detail=""):
    """Leave a breadcrumb in the output directory.

    The kernel's execution log is not always reachable - on a restricted
    network the log endpoint can be blocked while the output file *listing*
    still comes through. These markers make that bare listing enough to tell
    how far the run got.
    """
    path = PROGRESS / f"{step}.txt"
    path.write_text(str(detail), encoding="utf-8")
    print(f"[progress] {step} {detail}".rstrip(), flush=True)


def run(*args):
    """Run a step and stream its output.

    check=True alone would raise CalledProcessError showing only the command,
    while the child's diagnostics - which class had no annotations, how many
    cards could not be rectified - vanish. In Jupyter a subprocess inheriting
    the parent's stdout writes to the kernel's real stdout, which does not
    reach the cell output either. So the output is piped and echoed here.
    """
    label = args[1] if args and args[0] == "-m" else (args[0] if args else "step")
    print(">", " ".join(str(a) for a in args), flush=True)
    process = subprocess.Popen(
        [sys.executable, *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise SystemExit(f"{label} exited with code {code} - see the output above")
    mark(f"ok-{Path(label).stem}")


mark("01-source-ready", os.getcwd())


## 4. Get the data

With `DATA_SOURCE = "roboflow"` the dataset is pulled here, inside the kernel.
The API key comes from a Kaggle Secret, never from this notebook - a key
committed into a notebook is exactly how the previous one leaked.


In [ ]:
import os
from pathlib import Path

RAW = Path("/kaggle/working/data/raw")
NORMALISED = Path("/kaggle/working/data/normalised")
RECTIFIED = Path("/kaggle/working/data/rectified")

if DATA_SOURCE == "roboflow":
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        print("Roboflow key loaded from Kaggle Secrets")
    except Exception as exc:
        raise SystemExit(
            "Could not read the ROBOFLOW_API_KEY secret: " + str(exc) + "\n"
            "Add it under Add-ons > Secrets, then re-run this cell."
        ) from exc

    run("-m", "scripts.roboflow_pull",
        "--workspace", RF_WORKSPACE,
        "--project", RF_PROJECT,
        "--version", str(RF_VERSION),
        "--format", RF_FORMAT,
        "--profile", PROFILE,
        "--raw", str(RAW),
        "--out", str(NORMALISED))
    mark("02-roboflow-pulled")
    dataset_dir = NORMALISED
else:
    dataset_dir = Path(ATTACHED_DATA_YAML).parent
    print("using attached dataset at", dataset_dir)

if RECTIFY:
    run("-m", "scripts.rectify_dataset",
        "--src", str(dataset_dir),
        "--out", str(RECTIFIED),
        "--profile", PROFILE)
    mark("03-rectified")
    DATA_YAML = str(RECTIFIED / "data.yaml")
else:
    DATA_YAML = str(dataset_dir / "data.yaml")

print("\nDATA_YAML =", DATA_YAML)


## 5. Fail fast on a class-order mismatch

A reordered class list is silent and catastrophic: the pipeline would write the
father's name into the national id field and every downstream check would still
pass. Better to find out now than after a long training run.


In [ ]:
import yaml

from kyc.modules.ocr.profiles import get_profile

profile = get_profile(PROFILE)
expected = list(profile.class_names)

with open(DATA_YAML, encoding="utf-8") as handle:
    spec = yaml.safe_load(handle)

names = spec["names"]
actual = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)

print("profile :", expected)
print("dataset :", actual)

if actual != expected:
    raise SystemExit(
        "Class order mismatch. Fix the names: list in data.yaml to match the "
        "profile exactly, then re-run this cell."
    )
print()
print("OK - class order matches. Model will be saved as:", profile.detector_model)
mark("04-class-order-ok")


## 6. Train and export

In [ ]:
# run() streams the child's output; a bare subprocess.run would hide the
# training log behind a CalledProcessError.
run("training/kaggle/train_field_detector.py",
    "--data", DATA_YAML,
    "--profile", PROFILE,
    "--model", BASE,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--device", "0",
    "--out", "/kaggle/working/models")
mark("05-trained")


## 7. Verify the artifact

In [ ]:
import hashlib
from pathlib import Path

out = Path("/kaggle/working/models")
for path in sorted(out.glob("*")):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name:<40} {path.stat().st_size/1e6:7.1f} MB  sha256:{digest}")

print()
print("Record the sha256 in models/manifest.yaml with the dataset and commit.")
print("Then, in the repo:  python -m scripts.check_models")


## 8. Next: the text recognisers

The detector alone makes the pipeline work with the pretrained `easyocr`
backend. To go faster and more accurate, bootstrap a labelled crop set from
easyocr output and train the CRNNs - see `training/kaggle/README.md`.
